#Initialization

In [0]:
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("load_type", "incremental")
load_type = dbutils.widgets.get("load_type")

#Read table from S3 bucket

In [0]:
landing_path = "s3://sportpro-db/orders/landing/"
processed_path = "s3://sportpro-db/orders/processed/"

In [0]:
df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{landing_path}/*.csv")
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)


#Full Load

In [0]:
def full_load():
    df.write \
        .format("delta") \
        .mode("append") \
        .option("delta.enableChangeDataFeed", "true") \
        .saveAsTable("pcat.bronze.orders")

    # Moving files from source to processed directory
    files = dbutils.fs.ls(landing_path)
    for file in files:
        dbutils.fs.mv(
            file.path,
            f"{processed_path}/{file.name}",
            True
        )

#Incremental Load

In [0]:
def incremental_load():
    df.write \
        .format("delta") \
        .mode("append") \
        .option("delta.enableChangeDataFeed", "true") \
        .saveAsTable("pcat.bronze.orders")
    
    # Staging table to process the arrived incremenal data
    df.write \
     .format("delta")  \
     .option("delta.enableChangeDataFeed", "true") \
     .mode("overwrite") \
     .saveAsTable("pcat.bronze.staging_orders")

    # Moving files from source to processed directory
    files = dbutils.fs.ls(landing_path)
    for file in files:
        dbutils.fs.mv(
            file.path,
            f"{processed_path}/{file.name}",
            True
        )

#Orchestrate the loading process

In [0]:
if load_type == "full":
    print("performing bronze full load ----")
    full_load()

elif load_type == "incremental":
    print("performing bronze incremental load ----")
    incremental_load()